In [19]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [20]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.silver')

DataFrame[]

In [21]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType, ArrayType

bs4_parse_schema = StructType([
    StructField('clean_text', StringType(), True),
    StructField('clean_char_count', LongType(), True),
    StructField('code_block_count', LongType(), True),
    StructField('link_count', LongType(), True),
    StructField('tag_count', LongType(), True),
    StructField('is_xml_like', BooleanType(), True),
])

def _bs4_parse_text(raw_text):
    import re
    import html
    text = '' if raw_text is None else str(raw_text)
    trimmed = text.lstrip()
    is_xml_like = trimmed.startswith('<?xml') or trimmed.startswith('<rss') or trimmed.startswith('<feed') or trimmed.startswith('<xml')

    try:
        from bs4 import BeautifulSoup as _BeautifulSoup
        soup = _BeautifulSoup(text, 'html.parser')
        clean_text = ' '.join(soup.get_text(' ', strip=True).split())
        code_block_count = len(soup.find_all('code')) + len(soup.find_all('pre'))
        link_count = len(soup.find_all('a'))
        tag_count = len(soup.find_all(True))
    except Exception:
        code_block_count = len(re.findall(r'(?is)<\s*(pre|code)\b', text))
        link_count = len(re.findall(r'(?is)<\s*a\b[^>]*href\s*=', text))
        tag_count = len(re.findall(r'(?is)<\s*/?\s*[a-zA-Z][^>]*>', text))
        text_no_tags = re.sub(r'(?is)<[^>]+>', ' ', text)
        clean_text = ' '.join(html.unescape(text_no_tags).split())

    return (clean_text, int(len(clean_text)), int(code_block_count), int(link_count), int(tag_count), bool(is_xml_like))

bs4_parse_text_udf = F.udf(_bs4_parse_text, bs4_parse_schema)

# Read bronze source tables
bronze_questions = spark.table('ddc_databricks.bronze.so_questions')
bronze_answers = spark.table('ddc_databricks.bronze.so_answers')
bronze_tag_info = spark.table('ddc_databricks.bronze.so_tag_info')

In [22]:
# 1) Questions -> silver.so_questions
questions_flat = (
    bronze_questions
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('questions').alias('q')
    )
    .withColumn('q_json', F.to_json(F.col('q')))
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('q.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('q.creation_date')).cast('timestamp').alias('question_created_at'),
        F.from_unixtime(F.col('q.last_activity_date')).cast('timestamp').alias('question_last_activity_at'),
        F.from_unixtime(F.col('q.last_edit_date')).cast('timestamp').alias('question_last_edit_at'),
        F.from_unixtime(F.col('q.closed_date')).cast('timestamp').alias('question_closed_at'),
        F.col('q.score').cast('long').alias('question_score'),
        F.col('q.view_count').cast('long').alias('question_view_count'),
        F.col('q.answer_count').cast('long').alias('question_answer_count'),
        F.get_json_object(F.col('q_json'), '$.favorite_count').cast('long').alias('question_favorite_count'),
        F.col('q.is_answered').cast('boolean').alias('is_answered'),
        F.col('q.tags').alias('question_tags'),
        F.col('q.title').alias('question_title'),
        F.col('q.body').alias('question_body_html'),
        F.col('q.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('q.owner.display_name').alias('owner_display_name'),
        F.col('q.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('question_id').isNotNull())
    .withColumn('question_title_char_count', F.length(F.coalesce(F.col('question_title'), F.lit(''))))
    .withColumn('question_body_char_count', F.length(F.coalesce(F.col('question_body_html'), F.lit(''))))
    .withColumn('question_bs4', bs4_parse_text_udf(F.col('question_body_html')))
    .withColumn('question_body_text', F.col('question_bs4.clean_text'))
    .withColumn('question_body_text_char_count', F.col('question_bs4.clean_char_count'))
    .withColumn('question_body_code_block_count', F.col('question_bs4.code_block_count'))
    .withColumn('question_body_link_count', F.col('question_bs4.link_count'))
    .withColumn('question_body_html_tag_count', F.col('question_bs4.tag_count'))
    .withColumn('question_body_xml_like', F.col('question_bs4.is_xml_like'))
    .drop('question_bs4', 'question_body_html')
    .withColumn('question_tags_count', F.size(F.coalesce(F.col('question_tags'), F.array())))
)

questions_dedup_w = Window.partitionBy('package_name', 'question_id').orderBy(F.col('ingested_at').desc(), F.col('question_last_activity_at').desc())
questions_latest = (
    questions_flat
    .withColumn('rn', F.row_number().over(questions_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    questions_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_questions')
)

print('silver.so_questions created')

silver.so_questions created


In [23]:
# 2) Answers -> silver.so_answers
answers_flat = (
    bronze_answers
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.explode_outer('answers').alias('a')
    )
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.col('a.answer_id').cast('long').alias('answer_id'),
        F.col('a.question_id').cast('long').alias('question_id'),
        F.from_unixtime(F.col('a.creation_date')).cast('timestamp').alias('answer_created_at'),
        F.from_unixtime(F.col('a.last_activity_date')).cast('timestamp').alias('answer_last_activity_at'),
        F.from_unixtime(F.col('a.last_edit_date')).cast('timestamp').alias('answer_last_edit_at'),
        F.col('a.score').cast('long').alias('answer_score'),
        F.col('a.is_accepted').cast('boolean').alias('is_accepted'),
        F.col('a.body').alias('answer_body_html'),
        F.col('a.owner.user_id').cast('long').alias('owner_user_id'),
        F.col('a.owner.display_name').alias('owner_display_name'),
        F.col('a.owner.reputation').cast('long').alias('owner_reputation')
    )
    .filter(F.col('answer_id').isNotNull())
    .withColumn('answer_body_char_count', F.length(F.coalesce(F.col('answer_body_html'), F.lit(''))))
    .withColumn('answer_bs4', bs4_parse_text_udf(F.col('answer_body_html')))
    .withColumn('answer_body_text', F.col('answer_bs4.clean_text'))
    .withColumn('answer_body_text_char_count', F.col('answer_bs4.clean_char_count'))
    .withColumn('answer_body_code_block_count', F.col('answer_bs4.code_block_count'))
    .withColumn('answer_body_link_count', F.col('answer_bs4.link_count'))
    .withColumn('answer_body_html_tag_count', F.col('answer_bs4.tag_count'))
    .withColumn('answer_body_xml_like', F.col('answer_bs4.is_xml_like'))
    .drop('answer_bs4', 'answer_body_html')
)

answers_dedup_w = Window.partitionBy('package_name', 'answer_id').orderBy(F.col('ingested_at').desc(), F.col('answer_last_activity_at').desc())
answers_latest = (
    answers_flat
    .withColumn('rn', F.row_number().over(answers_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    answers_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_answers')
)

print('silver.so_answers created')

silver.so_answers created


In [24]:
# 3) Tag info -> silver.so_tag_info
tag_item_schema = ArrayType(StructType([
    StructField('name', StringType(), True),
    StructField('count', LongType(), True),
    StructField('has_synonyms', BooleanType(), True),
    StructField('is_moderator_only', BooleanType(), True),
    StructField('is_required', BooleanType(), True)
]))

tag_info_clean = (
    bronze_tag_info
    .select(
        F.col('_pypi_package').alias('package_name'),
        F.col('_so_tag').alias('so_tag'),
        F.to_timestamp('_ingested_at').alias('ingested_at'),
        F.col('_endpoint').alias('source_endpoint'),
        F.col('_source').alias('source_system'),
        F.to_json(F.col('data')).alias('data_json')
    )
    .withColumn('items_json', F.coalesce(F.get_json_object(F.col('data_json'), '$.items'), F.lit('[]')))
    .withColumn('items', F.from_json(F.col('items_json'), tag_item_schema))
    .withColumn('item', F.explode_outer('items'))
    .select(
        'package_name',
        'so_tag',
        'ingested_at',
        'source_endpoint',
        'source_system',
        F.coalesce(F.col('item.name'), F.col('so_tag')).alias('tag_name'),
        F.col('item.count').cast('long').alias('tag_question_count'),
        F.col('item.has_synonyms').cast('boolean').alias('has_synonyms'),
        F.col('item.is_moderator_only').cast('boolean').alias('is_moderator_only'),
        F.col('item.is_required').cast('boolean').alias('is_required')
    )
)

tag_info_dedup_w = Window.partitionBy('package_name', 'tag_name').orderBy(F.col('ingested_at').desc())
tag_info_latest = (
    tag_info_clean
    .withColumn('rn', F.row_number().over(tag_info_dedup_w))
    .filter(F.col('rn') == 1)
    .drop('rn')
)

(
    tag_info_latest
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_tag_info')
)

print('silver.so_tag_info created')

silver.so_tag_info created


In [25]:
# 4) Cross-linked package-level Stack Overflow features -> silver.so_package_features
question_features = (
    questions_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('question_id').alias('questions_total'),
        F.sum(F.when(F.col('is_answered') == F.lit(True), 1).otherwise(0)).alias('answered_questions_total'),
        F.avg('question_score').alias('avg_question_score'),
        F.avg('question_view_count').alias('avg_question_views'),
        F.avg('question_answer_count').alias('avg_answers_per_question'),
        F.sum(F.coalesce(F.col('question_favorite_count'), F.lit(0))).alias('question_favorites_total'),
        F.countDistinct('owner_user_id').alias('unique_question_askers'),
        F.max('question_created_at').alias('latest_question_at'),
        F.sum(F.when(F.col('question_created_at') >= F.expr('current_timestamp() - INTERVAL 30 DAYS'), 1).otherwise(0)).alias('questions_30d')
    )
    .withColumn('question_answered_rate', F.when(F.col('questions_total') > 0, F.col('answered_questions_total') / F.col('questions_total')).otherwise(F.lit(None)))
)

answer_features = (
    answers_latest
    .groupBy('package_name')
    .agg(
        F.countDistinct('answer_id').alias('answers_total'),
        F.sum(F.when(F.col('is_accepted') == F.lit(True), 1).otherwise(0)).alias('accepted_answers_total'),
        F.avg('answer_score').alias('avg_answer_score'),
        F.countDistinct('owner_user_id').alias('unique_answerers'),
        F.max('answer_created_at').alias('latest_answer_at'),
        F.sum(F.when(F.col('answer_created_at') >= F.expr('current_timestamp() - INTERVAL 30 DAYS'), 1).otherwise(0)).alias('answers_30d')
    )
    .withColumn('accepted_answer_rate', F.when(F.col('answers_total') > 0, F.col('accepted_answers_total') / F.col('answers_total')).otherwise(F.lit(None)))
)

tag_features = (
    tag_info_latest
    .groupBy('package_name')
    .agg(
        F.max('tag_question_count').alias('tag_total_questions'),
        F.max(F.when(F.col('has_synonyms') == F.lit(True), F.lit(1)).otherwise(F.lit(0))).alias('tag_has_synonyms_flag'),
        F.max(F.when(F.col('is_required') == F.lit(True), F.lit(1)).otherwise(F.lit(0))).alias('tag_is_required_flag')
    )
)

so_package_features = (
    question_features.alias('q')
    .join(answer_features.alias('a'), on='package_name', how='full')
    .join(tag_features.alias('t'), on='package_name', how='left')
    .withColumn('qa_ratio', F.when(F.col('questions_total') > 0, F.col('answers_total') / F.col('questions_total')).otherwise(F.lit(None)))
    .withColumn('community_activity_30d', F.coalesce(F.col('questions_30d'), F.lit(0)) + F.coalesce(F.col('answers_30d'), F.lit(0)))
)

(
    so_package_features
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_package_features')
)

print('silver.so_package_features created')

silver.so_package_features created


In [26]:
# 5) Basic data quality checks for Silver Stack Overflow outputs
quality_metrics = [
    ('bronze_so_questions_rows', bronze_questions.count()),
    ('bronze_so_answers_rows', bronze_answers.count()),
    ('bronze_so_tag_info_rows', bronze_tag_info.count()),
    ('silver_so_questions_rows', questions_latest.count()),
    ('silver_so_answers_rows', answers_latest.count()),
    ('silver_so_tag_info_rows', tag_info_latest.count()),
    ('silver_so_package_features_rows', so_package_features.count()),
    ('silver_so_questions_missing_clean_text', questions_latest.filter(F.col('question_body_text').isNull()).count()),
    ('silver_so_answers_missing_clean_text', answers_latest.filter(F.col('answer_body_text').isNull()).count()),
    ('silver_so_questions_xml_like', questions_latest.filter(F.col('question_body_xml_like') == F.lit(True)).count()),
    ('silver_so_answers_xml_like', answers_latest.filter(F.col('answer_body_xml_like') == F.lit(True)).count()),
    ('silver_so_unmatched_answers_to_questions', answers_latest.alias('a').join(questions_latest.alias('q'), on=[F.col('a.package_name') == F.col('q.package_name'), F.col('a.question_id') == F.col('q.question_id')], how='left').filter(F.col('q.question_id').isNull()).count())
]

quality_df = spark.createDataFrame(quality_metrics, ['metric_name', 'metric_value']).withColumn(
    'checked_at', F.current_timestamp()
)

(
    quality_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.silver.so_quality_checks')
)

quality_df.orderBy('metric_name').show(truncate=False)

+----------------------------------------+------------+--------------------------+
|metric_name                             |metric_value|checked_at                |
+----------------------------------------+------------+--------------------------+
|bronze_so_answers_rows                  |8           |2026-03-26 12:35:39.500179|
|bronze_so_questions_rows                |8           |2026-03-26 12:35:39.500179|
|bronze_so_tag_info_rows                 |8           |2026-03-26 12:35:39.500179|
|silver_so_answers_missing_clean_text    |0           |2026-03-26 12:35:39.500179|
|silver_so_answers_rows                  |1600        |2026-03-26 12:35:39.500179|
|silver_so_answers_xml_like              |0           |2026-03-26 12:35:39.500179|
|silver_so_package_features_rows         |8           |2026-03-26 12:35:39.500179|
|silver_so_questions_missing_clean_text  |0           |2026-03-26 12:35:39.500179|
|silver_so_questions_rows                |2400        |2026-03-26 12:35:39.500179|
|sil